# Nesterov accelerated gradient (NAG)
NAG by hand, its overshoot against momentum's, the narrow valley, a curve with a local minimum,
Keras' version of NAG, and momentum against NAG on MNIST.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## Momentum and NAG in one function
Momentum: v = beta v + eta grad(w).  NAG: v = beta v + eta grad(w - beta v), the gradient at the look-ahead point.
Then w = w - v for both.

In [ ]:
def path(grad, w0, eta, beta, nag, steps):
    w = np.array(w0, dtype=float)
    v = np.zeros_like(w)
    out = [w.copy()]
    for _ in range(steps):
        g = grad(w - beta * v) if nag else grad(w)
        v = beta * v + eta * g
        w = w - v
        out.append(w.copy())
    return np.array(out)

bowl = lambda w: w                       # gradient of L(w) = w^2 / 2
print("momentum, w0..w2:", path(bowl, -10.0, 0.1, 0.9, False, 2))
print("NAG,      w0..w2:", path(bowl, -10.0, 0.1, 0.9, True, 2))

## Overshooting on the bowl L(w) = w^2 / 2: learning rate 0.1, beta 0.9

In [ ]:
for nag in (False, True):
    P = path(bowl, -10.0, 0.1, 0.9, nag, 100)
    settled = next(k for k in range(len(P)) if np.all(np.abs(P[k:]) < 0.1))
    print(f"{'NAG' if nag else 'momentum':8s}: furthest past the minimum {P.max():.2f}, "
          f"|w| < 0.1 for good from step {settled}")

## The narrow valley L = (w1^2 + 100 w2^2) / 2 from (-10, 0.4), learning rate 0.01, beta 0.9

In [ ]:
K = 100
valley = lambda w: np.array([w[0], K * w[1]])
for nag in (False, True):
    P = path(valley, [-10.0, 0.4], 0.01, 0.9, nag, 600)
    loss = 0.5 * (P[:, 0] ** 2 + K * P[:, 1] ** 2)
    print(f"{'NAG' if nag else 'momentum':8s}: loss < 0.01 after {int(np.argmax(loss < 0.01))} steps, "
          f"furthest past the minimum along w1: {P[:, 0].max():.2f}, largest |w2| after step 10: {np.abs(P[10:, 1]).max():.4f}")

## A curve with a local minimum: L(w) = (w^2 - 4)^2 / 8 - 0.6 w
Twelve settings (two starts, three learning rates, two betas): which runs end in the global minimum near 2.14?

In [ ]:
df = lambda w: w * (w ** 2 - 4) / 2 - 0.6
rows = []
for start in (-3.0, -3.2):
    for eta in (0.02, 0.05, 0.1):
        for beta in (0.8, 0.9):
            ends = {name: path(df, start, eta, beta, nag, 300)[-1] for name, nag in (("momentum", False), ("NAG", True))}
            rows.append(dict(start=start, eta=eta, beta=beta, momentum_end=round(ends["momentum"], 2),
                             nag_end=round(ends["NAG"], 2)))
table = pd.DataFrame(rows)
print(table)
print("reach the global minimum: momentum", int((table.momentum_end > 0).sum()), "of 12, NAG",
      int((table.nag_end > 0).sum()), "of 12")
print("NAG escapes where momentum does not:", int(((table.nag_end > 0) & (table.momentum_end < 0)).sum()))

## Keras: SGD(momentum=0.9, nesterov=True)
Keras stores the look-ahead point w - beta v as its weight. Check on the bowl: Keras' weights
should equal our look-ahead points.

In [ ]:
w_k = keras.Variable(-10.0)
opt = keras.optimizers.SGD(learning_rate=0.1, momentum=0.9, nesterov=True)
keras_path = [float(w_k.numpy())]
for _ in range(5):
    opt.apply_gradients([(w_k.value * 1.0, w_k)])     # gradient of w^2/2 is w
    keras_path.append(float(w_k.numpy()))
P = path(bowl, -10.0, 0.1, 0.9, True, 6)
w, v = -10.0, 0.0
look = [w]
for _ in range(5):
    v = 0.9 * v + 0.1 * (w - 0.9 * v)
    w = w - v
    look.append(w - 0.9 * v)                            # the next look-ahead point
print("Keras weights:       ", np.round(keras_path, 4))
print("our look-ahead points:", np.round(look, 4))

## MNIST: momentum against NAG
10,000 training images, the 10,000 test images for validation, learning rate 0.01, batch size 64,
20 epochs, beta 0.9 and 0.99, three seeds each. Only `nesterov` changes.

In [ ]:
(x, y), (x_test, y_test) = keras.datasets.mnist.load_data()
x = x[:10000].reshape(-1, 784).astype("float32") / 255
y = y[:10000]
x_test = x_test.reshape(-1, 784).astype("float32") / 255

def train(beta, nesterov, seed, epochs=20):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([keras.Input(shape=(784,)),
                              keras.layers.Dense(128, activation="relu"),
                              keras.layers.Dense(64, activation="relu"),
                              keras.layers.Dense(10, activation="softmax")])
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=beta, nesterov=nesterov),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model.fit(x, y, epochs=epochs, batch_size=64, validation_data=(x_test, y_test), verbose=0).history

rows = []
for beta in (0.9, 0.99):
    for nesterov in (False, True):
        for seed in range(3):
            h = train(beta, nesterov, seed)
            for e in range(len(h["loss"])):
                rows.append(dict(beta=beta, nag=nesterov, seed=seed, epoch=e + 1, loss=h["loss"][e],
                                 val_accuracy=h["val_accuracy"][e]))
            print(f"beta {beta} nesterov {nesterov!s:5s} seed {seed}: final loss {h['loss'][-1]:.4f}, "
                  f"validation accuracy {h['val_accuracy'][-1]:.4f}")
runs = pd.DataFrame(rows)
runs.to_csv("data/mnist_nag.csv", index=False)

In [ ]:
mean = runs.groupby(["beta", "nag", "epoch"])[["loss", "val_accuracy"]].mean()
for beta in (0.9, 0.99):
    for nag in (False, True):
        m = mean.loc[(beta, nag)]
        ups = int((m["loss"].diff() > 0).sum())       # epochs where the mean training loss went up
        print(f"beta {beta} {'NAG     ' if nag else 'momentum'}: loss epoch 5 {m['loss'][5]:.4f}, epoch 20 "
              f"{m['loss'][20]:.4f}, val accuracy {m['val_accuracy'][20]:.4f}, epochs where the loss rose: {ups}")